In [1]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the base model
model_path = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_path)

# Set device based on availability of GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path, 
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
    trust_remote_code=True
)

# Load the adapter
model_peft = PeftModel.from_pretrained(model, "azam25/TinyLlama_instruct_generation")

c:\Users\elpulpo\Documents\Projects\COURS SIMPLON\AppConvHugFace\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
def generate_response(message, model, max_tokens=50):

    # Create prompt and encode input
    input = [
        {"role": "user", "content": message},
    ]
    prompt = tokenizer.apply_chat_template(input, tokenize=False) 
    encoded_input = tokenizer(prompt, return_tensors="pt", add_special_tokens=True) 

    # Ensure the input tensors are on the correct device (same as model)
    model_inputs = {key: value.to(device) for key, value in encoded_input.items()}  # Move inputs to the same device

    # Generate response with a limit on the new tokens only (response tokens)
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    # Decode output
    decoded_output = tokenizer.batch_decode(generated_ids) 

    # Extract assistant's response
    assistant_response = decoded_output[0].split("<|assistant|>")[1].split("</s>")[0].strip()

    return assistant_response


In [13]:
message = "Hello, who are you?"

# Generate and print response
response = generate_response(message, model)
print(response)

I am a computer program.


In [19]:
message = "Cool, how are you?"

# Generate and print response
response = generate_response(message, model)
print(response)

I'm doing well, how are you?
